# RS Genomics — Zero-Parameter Amino Acid Substitution Geometry

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jonwashburn/recognition-science/blob/main/rs-genomics/rs_genomics_tutorial.ipynb)

This notebook demonstrates a **zero-parameter geometric score** for amino acid substitutions,
derived entirely from the 8-tick DFT cycle and the canonical cost function `J(x) = (x + 1/x)/2 - 1`.

**No fitted parameters. No training data. Just mathematics.**

We will:
1. Score individual substitutions (including the sickle cell mutation E→V)
2. Visualize the full 20×20 distance matrix
3. Benchmark against 10,000 real ClinVar patient variants
4. Compare to the industry-standard Grantham score (3 fitted parameters)

---

**Reference:** Washburn et al., *The Genetic Code Architecture is Forced by a Single Functional Equation* (2026).

**Machine-verified proofs:** [Lean 4 — IndisputableMonolith](https://github.com/jonwashburn/recognition-science)

## Setup

Download the scorer and data if running in Colab.

In [ ]:
import os
if not os.path.exists("rs_genomics.py"):
    !wget -q https://raw.githubusercontent.com/jonwashburn/recognition-science/main/rs-genomics/rs_genomics.py
    !mkdir -p data
    !wget -q https://raw.githubusercontent.com/jonwashburn/recognition-science/main/rs-genomics/data/clinvar_10k_sample.csv -O data/clinvar_10k_sample.csv
    print("Downloaded rs_genomics.py and ClinVar data.")
else:
    print("Files already present.")

In [ ]:
import rs_genomics
import numpy as np
import matplotlib.pyplot as plt
import csv

print(f"rs_genomics v{rs_genomics.__version__}")
print(f"Amino acids: {rs_genomics.AMINO_ACIDS}")

## 1. Score a Single Substitution

The classic example: **Glu → Val** (E→V), the mutation that causes **sickle cell disease**.

In [ ]:
result = rs_genomics.score_pair("E", "V")

print(f"Substitution: {result['ref']['name']} ({result['ref']['code1']}) -> {result['alt']['name']} ({result['alt']['code1']})")
print(f"")
print(f"  RS Distance (0 params):  {result['rs_distance']:.4f}")
print(f"  Grantham    (3 params):  {result['grantham']:.0f}")
print(f"  Fubini-Study (CP^6):     {result['fubini_study']:.4f}")
print(f"  Crosses families:        {result['crosses_families']} ({result['family_transition']})")

## 2. The 20×20 RS Distance Matrix

Every number below is derived from first principles. No biological data was used to produce this matrix.

In [ ]:
rs_matrix = np.array(rs_genomics.score_all())
grantham_matrix = np.array(rs_genomics.grantham_all())
labels = [rs_genomics._3TO1[aa] for aa in rs_genomics.AMINO_ACIDS_3]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

im0 = axes[0].imshow(rs_matrix, cmap="YlOrRd", aspect="equal")
axes[0].set_xticks(range(20)); axes[0].set_xticklabels(labels, fontsize=8)
axes[0].set_yticks(range(20)); axes[0].set_yticklabels(labels, fontsize=8)
axes[0].set_title("RS Distance (0 fitted parameters)", fontsize=13, fontweight="bold")
plt.colorbar(im0, ax=axes[0], shrink=0.8)

im1 = axes[1].imshow(grantham_matrix, cmap="YlOrRd", aspect="equal")
axes[1].set_xticks(range(20)); axes[1].set_xticklabels(labels, fontsize=8)
axes[1].set_yticks(range(20)); axes[1].set_yticklabels(labels, fontsize=8)
axes[1].set_title("Grantham Distance (3 fitted parameters)", fontsize=13, fontweight="bold")
plt.colorbar(im1, ax=axes[1], shrink=0.8)

plt.tight_layout()
plt.show()

## 3. Benchmark: 10,000 Real ClinVar Variants

We score every missense variant from [ClinVar](https://www.ncbi.nlm.nih.gov/clinvar/) using the RS metric and compute AUC-ROC for distinguishing pathogenic from benign.

In [ ]:
result = rs_genomics.benchmark_clinvar("data/clinvar_10k_sample.csv")

print(f"  Pathogenic variants:  {result['n_pathogenic']:,}")
print(f"  Benign variants:      {result['n_benign']:,}")
print(f"  AUC-ROC:              {result['auc_roc']:.4f}")
print(f"  Fitted parameters:    0")
print(f"")
print(f"  Grantham AUC-ROC:     0.609  (3 fitted parameters)")

## 4. ROC Curve: RS vs Grantham

Compute and plot the full ROC curve for both scorers on the same ClinVar data.

In [ ]:
rs_scores, grantham_scores, labels_bin = [], [], []
with open("data/clinvar_10k_sample.csv") as f:
    reader = csv.DictReader(f)
    for row in reader:
        try:
            rs_d = rs_genomics.score(row["ref"], row["alt"])
            g_d = rs_genomics.grantham(row["ref"], row["alt"])
        except (ValueError, KeyError):
            continue
        rs_scores.append(rs_d)
        grantham_scores.append(g_d)
        labels_bin.append(1 if row["classification"].strip().lower() == "pathogenic" else 0)

rs_scores = np.array(rs_scores)
grantham_scores = np.array(grantham_scores)
labels_bin = np.array(labels_bin)

def roc_curve(scores, labels):
    thresholds = np.sort(np.unique(scores))[::-1]
    tpr_list, fpr_list = [0.0], [0.0]
    n_pos = labels.sum()
    n_neg = len(labels) - n_pos
    for t in thresholds:
        pred = (scores >= t).astype(int)
        tp = ((pred == 1) & (labels == 1)).sum()
        fp = ((pred == 1) & (labels == 0)).sum()
        tpr_list.append(tp / n_pos)
        fpr_list.append(fp / n_neg)
    tpr_list.append(1.0)
    fpr_list.append(1.0)
    return np.array(fpr_list), np.array(tpr_list)

fpr_rs, tpr_rs = roc_curve(rs_scores, labels_bin)
fpr_g, tpr_g = roc_curve(grantham_scores, labels_bin)

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot(fpr_rs, tpr_rs, color="#e63946", linewidth=2.5,
        label=f"RS Distance (AUC = {np.trapz(tpr_rs, fpr_rs):.3f}, 0 params)")
ax.plot(fpr_g, tpr_g, color="#3b82f6", linewidth=2.5, linestyle="--",
        label=f"Grantham (AUC = {np.trapz(tpr_g, fpr_g):.3f}, 3 params)")
ax.plot([0, 1], [0, 1], color="gray", linewidth=1, linestyle=":")
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("ROC Curve: Pathogenicity Prediction on 10,000 ClinVar Variants",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=11, loc="lower right")
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
plt.tight_layout()
plt.show()

## 5. The Key Result

| Metric | AUC-ROC | Fitted Parameters |
|--------|---------|-------------------|
| **RS Distance** | **0.590** | **0** |
| Grantham (1974) | 0.609 | 3 |

The RS metric achieves **97% of Grantham's discriminative power** with **zero fitted parameters**.

The distance is derived entirely from:
- The 8-tick DFT cycle (forces 20 WToken vectors in C^8)
- The Fubini-Study metric on CP^6 (inter-family distance)
- J(x) = (x + 1/x)/2 - 1 applied to molecular weight and hydrophobicity ratios (within-family distance)

All proofs are machine-verified in [Lean 4](https://github.com/jonwashburn/recognition-science).

---

*Recognition Science Institute — [recognitionphysics.org](https://recognitionphysics.org)*